# Ibovespa data cleaning and exploratory analysis

Daily `^BVSP` data run from 2000 through 2025. The main analytical window starts in 2008. The notebook preserves the raw input and records validation flags, missing sessions, and return outliers.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from limpeza_ibovespa import limpar_dados, exportar

INICIO = "2000-01-01"
FIM = "2026-01-01"  # exclusivo; período original preservado
PASTA = Path("dados")
PASTA.mkdir(exist_ok=True)
from retratos_locais import raiz_privada
CACHE = raiz_privada() / "dados/ibovespa_bruto.csv"
if not CACHE.is_file():
    raise FileNotFoundError(f"Base diária ausente: {CACHE}")

df_bruto = pd.read_csv(CACHE, index_col="Date", parse_dates=True)
print("Linhas brutas:", len(df_bruto))


## Cleaning, calendar, and audit

The notebook exports the audit tables after cleaning. `df_vol` retains the complete price sequence from 2008, including sessions with unavailable volume.

In [ ]:
resultado = limpar_dados(df_bruto, INICIO, FIM)
df = resultado["completo"].copy()
df_vol = resultado["analise"].copy()
exportar(resultado, PASTA)
print(json.dumps(resultado["resumo"], indent=2, ensure_ascii=False))
print("\nVolume por ano:")
display(resultado["volume_anual"])
print("\nSessões ausentes (revisar fonte/calendário):")
display(resultado["sessoes_ausentes"])
print("\nLinhas rejeitadas:")
display(resultado["rejeitadas"])
print("\nRetornos com módulo > 15% (sinalização, não exclusão):")
display(resultado["extremos"])


In [ ]:
print("Ausências na base de análise:")
display(df_vol.isna().sum().to_frame("ausentes"))
display(df_vol.head())
display(df_vol.tail())
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax1.plot(df_vol.index, df_vol.Close)
ax1.set(title="Ibovespa — preços preservados desde 2008", ylabel="Pontos", yscale="log")
ax2.plot(df_vol.index, df_vol.Volume)
ax2.set(ylabel="Volume informado pela fonte", xlabel="Data")
plt.tight_layout()
plt.show()


## Returns and the contiguous sample

Sequential analyses use the most recent uninterrupted run, starting at the first price after the last missing exchange session since 2008. The full audit preserves the gaps.

In [ ]:
lacunas = pd.DatetimeIndex(resultado["sessoes_ausentes"]["Date"])
lacunas = lacunas[lacunas>=pd.Timestamp("2008-01-01")]
if len(lacunas):
    df_vol = df_vol.loc[df_vol.index>lacunas.max()].copy()
if df_vol.empty:
    raise ValueError("Não há trecho observado após a última lacuna.")
df_vol.loc[df_vol.index[0],["ret_simples","ret_log"]] = np.nan
if df_vol.ret_simples.iloc[1:].isna().any():
    raise ValueError("O recorte contém lacuna interna.")
print(f"Trecho contínuo: {df_vol.index[0].date()} a {df_vol.index[-1].date()} | {len(df_vol)} pregões")
print(df_vol[["Close","ret_simples","ret_log"]].head())

## Forecast and execution horizons

The exploratory predictability study uses close-to-close returns. The walk-forward backtest enters at the next opening price and exits at the close of its configured holding period. A single-position rule applies across adjacent test windows.

In [ ]:
# 1. Relação matemática: ret_log = ln(1 + ret_simples)
diff = (df_vol["ret_log"] - np.log1p(df_vol["ret_simples"])).abs().max()
print("diferenca maxima:", diff)   # deve ser ~1e-16

# 2. Para retornos pequenos, os dois são quase iguais
print(df_vol[["ret_simples", "ret_log"]].describe())

# 3. Os dias mais extremos
print(df_vol[["Close", "ret_simples", "ret_log"]].nsmallest(5, "ret_simples"))
print(df_vol[["Close", "ret_simples", "ret_log"]].nlargest(5, "ret_simples"))

## Descriptive statistics

In [ ]:
# Configuração das estatísticas
r  = df_vol["ret_log"].dropna()       # log-return
rs = df_vol["ret_simples"].dropna()   # retorno simples
ANO = 252                             # pregões por ano

pd.options.display.float_format = "{:.6f}".format

In [ ]:
from scipy import stats
def desc_basica(x):
    qs = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    out = {
        "n": len(x),
        # tendência central
        "mean": x.mean(),
        "median": x.median(),
        "mode_aprox (arred. 0.1%)": x.round(3).mode().iloc[0],
        # dispersão
        "var": x.var(),
        "std": x.std(),
        "mad (mediana)": stats.median_abs_deviation(x),
        "mad_escalada (~std)": stats.median_abs_deviation(x, scale="normal"),
        "desvio_medio_abs": (x - x.mean()).abs().mean(),
        "min": x.min(),
        "max": x.max(),
        "amplitude": x.max() - x.min(),
        "iqr": x.quantile(0.75) - x.quantile(0.25),
    }
    for p in qs:
        out[f"q{round(p*100):02d}"] = x.quantile(p)

    # forma da distribuição (versões populacionais, bias=True)
    out["skewness"] = stats.skew(x, bias=True)
    out["kurtosis_excess"] = stats.kurtosis(x, fisher=True, bias=True)
    out["kurtosis_raw"] = out["kurtosis_excess"] + 3          # <- a do PSR/DSR
    # versões "ajustadas" do pandas, só para comparação
    out["skewness_pandas (ajustada)"] = x.skew()
    out["kurtosis_excess_pandas (ajustada)"] = x.kurt()
    return pd.Series(out)

tab_desc = pd.DataFrame({"log": desc_basica(r), "simples": desc_basica(rs)})
print(tab_desc.to_string())

In [ ]:
mu, sd = r.mean(), r.std()
z = (r - mu) / sd

# Quantos dias passam de k desvios padrão vs. o que uma normal prevê
caudas = []
for k in [2, 3, 4, 5]:
    obs = int((z.abs() > k).sum())
    esp = len(r) * 2 * stats.norm.sf(k)
    caudas.append({"k_sigma": k, "observados": obs, "esperado_normal": round(esp, 2),
                   "razao_obs/esp": obs / esp if esp > 0 else np.nan})
print(pd.DataFrame(caudas).to_string(index=False))

# Testes de normalidade (rejeitar H0 = não é normal)
jb = stats.jarque_bera(r)
dag = stats.normaltest(r)
print("\nJarque-Bera:", jb.statistic, "p =", jb.pvalue)
print("D'Agostino-Pearson:", dag.statistic, "p =", dag.pvalue)

# Assimetria das caudas
print("\nrazão q95/|q05|:", r.quantile(0.95) / abs(r.quantile(0.05)))
print("razão q99/|q01|:", r.quantile(0.99) / abs(r.quantile(0.01)))

In [ ]:
rf_diario = 0.0   # depois agnt tem que trocar pelo CDI diário para um Sharpe de verdade

# Retorno e volatilidade anualizados
ret_anual_aritm = rs.mean() * ANO
vol_anual = rs.std() * np.sqrt(ANO)

# Curva de capital e drawdown
equity = (1 + rs).cumprod()
cagr = equity.iloc[-1] ** (ANO / len(rs)) - 1
pico = equity.cummax().clip(lower=1.0)
dd = equity / pico - 1
max_dd = dd.min()

# Duração máxima do drawdown (dias de pregão abaixo do pico anterior)
sub = dd < 0
grp = (sub != sub.shift()).cumsum()
max_dd_dur = int(sub.groupby(grp).sum().max())

# Razões
sharpe = (rs.mean() - rf_diario) / rs.std() * np.sqrt(ANO)
downside = np.sqrt((np.minimum(rs - rf_diario, 0) ** 2).mean())
sortino = (rs.mean() - rf_diario) / downside * np.sqrt(ANO)
calmar = cagr / abs(max_dd)

metricas = pd.Series({
    "retorno_anual_aritmetico": ret_anual_aritm,
    "CAGR": cagr,
    "vol_anual": vol_anual,
    "sharpe_anual": sharpe,
    "sharpe_diario (p/ PSR)": rs.mean() / rs.std(),
    "downside_dev_diario": downside,
    "sortino_anual": sortino,
    "max_drawdown": max_dd,
    "data_max_drawdown": dd.idxmin(),
    "max_dd_duracao_dias": max_dd_dur,
    "calmar": calmar,
    "retorno_acumulado": equity.iloc[-1] - 1,
})
print(metricas.to_string())

In [ ]:
def var_cvar(x, nivel):
    q = x.quantile(1 - nivel)
    return -q, -x[x <= q].mean()

# Histórico
for nv in [0.95, 0.99]:
    v, c = var_cvar(rs, nv)
    print(f"Histórico {int(nv*100)}%:  VaR = {v:.4%}   CVaR/ES = {c:.4%}")

# Paramétrico (assume normal)
for nv in [0.95, 0.99]:
    zn = stats.norm.ppf(1 - nv)
    print(f"Normal    {int(nv*100)}%:  VaR = {-(rs.mean() + rs.std()*zn):.4%}")

# Cornish-Fisher (corrige assimetria e curtose)
S = stats.skew(rs, bias=True)
K = stats.kurtosis(rs, fisher=True, bias=True)   # excesso
for nv in [0.95, 0.99]:
    zn = stats.norm.ppf(1 - nv)
    zcf = (zn + (zn**2 - 1)*S/6 + (zn**3 - 3*zn)*K/24 - (2*zn**3 - 5*zn)*(S**2)/36)
    print(f"Cornish-Fisher {int(nv*100)}%: VaR = {-(rs.mean() + rs.std()*zcf):.4%}")

In [ ]:
def max_streak(cond):
    g = (cond != cond.shift()).cumsum()
    return int(cond.groupby(g).sum().max())

ganhos = rs[rs > 0]
perdas = rs[rs < 0]

perfil = pd.Series({
    "dias_positivos": len(ganhos),
    "dias_negativos": len(perdas),
    "dias_zero": int((rs == 0).sum()),
    "taxa_acerto (win rate)": len(ganhos) / len(rs),
    "ganho_medio": ganhos.mean(),
    "perda_media": perdas.mean(),
    "payoff (ganho_medio/|perda_media|)": ganhos.mean() / abs(perdas.mean()),
    "profit_factor": ganhos.sum() / abs(perdas.sum()),
    "expectativa_por_dia": rs.mean(),
    "melhor_dia": rs.max(),
    "data_melhor_dia": rs.idxmax(),
    "pior_dia": rs.min(),
    "data_pior_dia": rs.idxmin(),
    "maior_sequencia_alta": max_streak(rs > 0),
    "maior_sequencia_baixa": max_streak(rs < 0),
})
print(perfil.to_string())

# Contribuição dos extremos: o que acontece se perder os N melhores dias?
for n in [5, 10, 20]:
    sem = rs.drop(rs.nlargest(n).index)
    print(f"Sem os {n} melhores dias: retorno acumulado = {(1+sem).prod()-1:.2%}  "
          f"(original: {(1+rs).prod()-1:.2%})")

In [ ]:
from statsmodels.tsa.stattools import acf, adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

lags = 10
tab_acf = pd.DataFrame({
    "acf_retorno": acf(r, nlags=lags)[1:],
    "acf_retorno^2": acf(r**2, nlags=lags)[1:],
    "acf_|retorno|": acf(r.abs(), nlags=lags)[1:],
}, index=range(1, lags + 1))
tab_acf.index.name = "lag"
print(tab_acf.to_string())
print("\nbanda de ~95% sob ruído branco: ±", 1.96 / np.sqrt(len(r)))

print("\nLjung-Box (retornos):")
print(acorr_ljungbox(r, lags=[5, 10, 20], return_df=True))
print("\nLjung-Box (retornos^2):")
print(acorr_ljungbox(r**2, lags=[5, 10, 20], return_df=True))

adf = adfuller(r)
print("\nADF (estacionariedade): estat =", adf[0], " p =", adf[1])

arch = het_arch(r, nlags=10)
print("Teste ARCH (efeito de volatilidade): LM =", arch[0], " p =", arch[1])

In [ ]:
# Por ano
por_ano = pd.DataFrame({
    "retorno_anual": (1 + rs).groupby(rs.index.year).prod() - 1,
    "vol_anual": rs.groupby(rs.index.year).std() * np.sqrt(ANO),
    "sharpe": rs.groupby(rs.index.year).mean() / rs.groupby(rs.index.year).std() * np.sqrt(ANO),
    "skew": rs.groupby(rs.index.year).apply(lambda x: stats.skew(x, bias=True)),
    "kurt_raw": rs.groupby(rs.index.year).apply(lambda x: stats.kurtosis(x, fisher=False, bias=True)),
    "pior_dia": rs.groupby(rs.index.year).min(),
    "melhor_dia": rs.groupby(rs.index.year).max(),
    "taxa_acerto": rs.groupby(rs.index.year).apply(lambda x: (x > 0).mean()),
})
print(por_ano.to_string())

# Por dia da semana
ordem = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
por_dia = rs.groupby(rs.index.day_name()).agg(["count", "mean", "std", "median"]).reindex(ordem)
por_dia["taxa_acerto"] = rs.groupby(rs.index.day_name()).apply(lambda x: (x > 0).mean()).reindex(ordem)
print("\n", por_dia.to_string())

# Por mês (sazonalidade)
por_mes = rs.groupby(rs.index.month).agg(["count", "mean", "std"])
por_mes["taxa_acerto"] = rs.groupby(rs.index.month).apply(lambda x: (x > 0).mean())
print("\n", por_mes.to_string())

In [ ]:
# ANOVA: o retorno médio difere entre os dias da semana?
grupos = [g.values for _, g in rs.groupby(rs.index.dayofweek)]
print(stats.f_oneway(*grupos))
print(stats.kruskal(*grupos))   # versão não paramétrica

In [ ]:
vol21  = rs.rolling(21).std()  * np.sqrt(ANO)
vol63  = rs.rolling(63).std()  * np.sqrt(ANO)
vol252 = rs.rolling(252).std() * np.sqrt(ANO)

print(pd.DataFrame({"vol21": vol21, "vol63": vol63, "vol252": vol252}).describe().to_string())

# Volatilidade ponderada exponencialmente (estilo RiskMetrics, lambda = 0.94)
vol_ewma = np.sqrt((rs**2).ewm(alpha=1 - 0.94, adjust=False).mean()) * np.sqrt(ANO)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 13))

# 1. Histograma vs normal
ax = axes[0, 0]
ax.hist(r, bins=120, density=True, alpha=0.6, color="tab:blue")
xs = np.linspace(r.min(), r.max(), 500)
ax.plot(xs, stats.norm.pdf(xs, r.mean(), r.std()), "r-", label="Normal")
ax.set_title("Histograma dos log-returns vs normal")
ax.legend()

# 2. Mesmo histograma em escala log (mostra as caudas)
ax = axes[0, 1]
ax.hist(r, bins=120, density=True, alpha=0.6, color="tab:blue")
ax.plot(xs, stats.norm.pdf(xs, r.mean(), r.std()), "r-")
ax.set_yscale("log")
ax.set_ylim(1e-2)
ax.set_title("Mesmo gráfico, eixo y em log (caudas)")

# 3. QQ-plot
stats.probplot(r, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("QQ-plot vs normal")

# 4. Retornos diários (clusters de volatilidade)
axes[1, 1].plot(r.index, r, lw=0.5)
axes[1, 1].set_title("Log-returns diários")

# 5. Volatilidade móvel
axes[2, 0].plot(vol21, label="21d", lw=0.8)
axes[2, 0].plot(vol63, label="63d", lw=0.8)
axes[2, 0].plot(vol252, label="252d", lw=1.2)
axes[2, 0].set_title("Volatilidade anualizada móvel")
axes[2, 0].legend()

# 6. Drawdown
axes[2, 1].fill_between(dd.index, dd, 0, color="tab:red", alpha=0.5)
axes[2, 1].set_title("Drawdown")

plt.tight_layout()
plt.show()

# ACF de retornos e retornos^2
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(r, lags=40, ax=ax[0], title="ACF dos retornos")
plot_acf(r**2, lags=40, ax=ax[1], title="ACF dos retornos ao quadrado")
plt.tight_layout()
plt.show()

In [ ]:
from backtest_legado import declust, walk_forward

# parametros
LIM_QUEDA     = -0.02
PCT_VOL       = 0.90
JANELA_VOL    = 21
MIN_HIST      = 504
N_BOOT        = 10000

HORIZONTE     = 5
MIN_EVENTOS   = 30
ALPHA         = 0.05
SEED          = 42

JANELA_TREINO = 756
JANELA_TESTE  = 252

BOOT_BLOCK    = 10
CUSTO_BPS     = 10      # custo POR LADO (entrada e saída); ida e volta = 2x


rng = np.random.default_rng(SEED)
custo_rt = 2 * CUSTO_BPS / 1e4

close = df_vol["Close"]
ret = close.pct_change(fill_method=None)
vol = ret.rolling(JANELA_VOL).std() * np.sqrt(ANO)

# Limite de vol usando SÓ o passado (expanding, deslocado 1 dia) -> sem look-ahead
lim_vol = vol.expanding(min_periods=MIN_HIST).quantile(PCT_VOL).shift(1)
amostra = lim_vol.notna() & vol.notna()          # mesma amostra para todas as condições
vol_alta = vol >= lim_vol
queda = ret < LIM_QUEDA

conds = {
    "Todos (amostra)":                   amostra,
    f"Queda < {LIM_QUEDA:.0%}":          amostra & queda,
    f"Vol alta (top {1-PCT_VOL:.0%})":   amostra & vol_alta,
    "Queda E vol alta":                  amostra & queda & vol_alta,
}
N_TESTES = (len(conds) - 1) * 2                  # 3 condições x 2 horizontes (Bonferroni)

def retorno_previsao(h):
    return close.shift(-h) / close - 1           # retorno de t (fechamento) até t+h

def boot_diff(f, m, chunk=500):
    """Bootstrap em blocos: reamostra (retorno_futuro, condição) JUNTOS, preservando
    a dependência temporal. Devolve a distribuição de E[cond] - E[base]."""
    n = len(f); B = BOOT_BLOCK
    nb = int(np.ceil(n / B)); ar = np.arange(B)
    out = []
    for _ in range(int(np.ceil(N_BOOT / chunk))):
        starts = rng.integers(0, n - B + 1, size=(chunk, nb))
        idx = (starts[:, :, None] + ar).reshape(chunk, -1)[:, :n]
        ff, mm = f[idx], m[idx]
        cnt = mm.sum(axis=1)
        cm = np.where(cnt > 0, (ff * mm).sum(axis=1) / np.maximum(cnt, 1), np.nan)
        out.append(cm - ff.mean(axis=1))
    d = np.concatenate(out)[:N_BOOT]
    return d[~np.isnan(d)]


# PARTE 1: E[R_{t+h} | X_t] na amostra inteira

def analisar(h):
    f_all = retorno_previsao(h)
    ok = (amostra & f_all.notna()).values
    idx_ok = df_vol.index[ok]
    f = f_all.values[ok]
    base = f.mean()
    cols = {}
    for nome, m in conds.items():
        mk = m.values[ok]
        pos = np.flatnonzero(mk)
        n = len(pos)
        if n < 2:
            cols[nome] = pd.Series({"n_dias": n}); continue
        x = f[pos]
        nd = declust(pos, h)
        epis = int((pd.Series(idx_ok[pos]).diff().dt.days > 10).sum() + 1)

        if nome.startswith("Todos"):
            diff, lo, hi, p, status = 0.0, np.nan, np.nan, np.nan, "(referência)"
        else:
            d = boot_diff(f, mk)
            lo, hi = np.percentile(d, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
            p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
            diff = x.mean() - base
            if len(nd) < MIN_EVENTOS:
                status = "INSUFICIENTE (< MIN_EVENTOS)"
            elif p < ALPHA / N_TESTES:
                status = "significativo (Bonferroni)"
            elif p < ALPHA:
                status = "significativo SEM correção"
            else:
                status = "indistinguível de ruído"

        cols[nome] = pd.Series({
            "n_dias": n,
            "n_nao_sobrepostos": len(nd),
            "episodios (gap>10d)": epis,
            f"E[R_t+{h}]": x.mean(),
            "mediana": np.median(x),
            "desvio": x.std(),
            "E[R] base (amostra)": base,
            "diferenca vs base": diff,
            "IC diff (boot blocos) inf": lo,
            "IC diff (boot blocos) sup": hi,
            "p_boot (bilateral)": p,
            "prob(R>0)": (x > 0).mean(),
            "pior": x.min(),
            "melhor": x.max(),
            "veredito": status,
        })
    return pd.DataFrame(cols)

pd.options.display.float_format = "{:.5f}".format
for h in sorted({1, HORIZONTE}):
    print(f"\n{'='*70}\nHORIZONTE = {h} dia(s)   |   amostra: {amostra.sum()} dias\n{'='*70}")
    print(analisar(h).to_string())


# PARTE 2: Walk-forward (treino -> sinal -> teste, com custo)

SAIDA_LEGADO = Path("resultados_legado")/"P1_execucao_D1"
SAIDA_LEGADO.mkdir(parents=True,exist_ok=True)
(SAIDA_LEGADO/"configuracao.json").write_text(json.dumps({
    "instrumento":"^BVSP","tipo_preco":"referencia do indice",
    "inicio_recorte":str(df_vol.index[0].date()),"fim_recorte":str(df_vol.index[-1].date()),
    "treino":JANELA_TREINO,"teste":JANELA_TESTE,"horizonte":HORIZONTE,"custo_bps_lado":CUSTO_BPS,
    "entrada":"abertura t+1","saida":"fechamento t+h"},indent=2),encoding="utf-8")

for nome, usar_vol in [("Queda E vol alta", True), (f"Só queda < {LIM_QUEDA:.0%}", False)]:
    folds, trades = walk_forward(
        df_vol, ret, vol, horizonte=HORIZONTE, janela_treino=JANELA_TREINO,
        janela_teste=JANELA_TESTE, lim_queda=LIM_QUEDA, pct_vol=PCT_VOL,
        min_eventos=MIN_EVENTOS, custo_rt=custo_rt, usar_vol=usar_vol,
    )
    etiqueta = "queda_e_vol" if usar_vol else "queda"
    folds.to_csv(SAIDA_LEGADO/f"janelas_{etiqueta}.csv",index=False)
    trades.to_csv(SAIDA_LEGADO/f"trades_{etiqueta}.csv",index=False)
    print(f"\n{'='*70}\nWALK-FORWARD ({nome}) | h = {HORIZONTE} | custo ida+volta = {custo_rt*1e4:.0f} bps\n{'='*70}")
    print(folds.to_string(index=False))
    if len(trades) < 2:
        print("-> Sem operações suficientes (o filtro MIN_EVENTOS barrou o sinal nos treinos).")
    else:
        L = trades["liquido"]
        t = L.mean() / (L.std() / np.sqrt(len(L)))
        print(f"\ntrades: {len(L)} | médio bruto: {trades['bruto'].mean():.4%} | "
              f"médio líquido: {L.mean():.4%} | acerto: {(L>0).mean():.1%} | t(líq vs 0): {t:.2f} | "
              f"soma líquida: {L.sum():.2%}")